In [ ]:
SMOKE = True   # True = smoke run: 8 pairs per scoring set, 3 warm-up + 8 latency samples, 2 throughput
               # batches, every output under /kaggle/working/smoke/, every number DISCARDED.
               # Set to False for the full run (docs/phase3-serving-benchmark-runbook.md).
# PricePilot Phase 3 item 8 -- serving benchmark. Protocol: docs/phase3-serving-benchmark-protocol.md
# (pre-registered). This notebook computes NO F1 and reads NO TEST labels: it only writes scores,
# latencies and facts. Thresholds and scoring happen locally, in session 2.
N_SET = 8 if SMOKE else None          # pairs per scoring set (None = all)
N_WARMUP = 3 if SMOKE else 10         # untimed warm-up pairs before the latency pass
N_LAT = 8 if SMOKE else 287           # latency samples (protocol 5.5: the 287 TEST pairs)
N_THR_BATCHES = 2 if SMOKE else 10    # throughput: batches of 16 (protocol 5.5: 160 pairs)
THR_BATCH = 16

In [ ]:
import ast, gc, glob, hashlib, json, os, platform, re, resource, shutil, subprocess, sys, time, traceback
from pathlib import Path

WORK = Path("/kaggle/working")
OUT = WORK / "smoke" if SMOKE else WORK
TMP = (Path("/kaggle/temp") if Path("/kaggle/temp").exists() else Path("/tmp")) / ("pp-smoke" if SMOKE else "pp-full")
OUT.mkdir(parents=True, exist_ok=True)
TMP.mkdir(parents=True, exist_ok=True)   # ONNX graphs live here, NOT in the saved output

r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime", "onnxscript", "peft"],
                   capture_output=True, text=True)
print(r.stdout[-400:], r.stderr[-400:])
assert r.returncode == 0, "pip install failed"

import numpy as np
import torch
import transformers
import onnx, onnxruntime, peft, tokenizers

def jdump(obj, name, folder=None):
    path = (folder or OUT) / name
    path.write_text(json.dumps(obj, indent=1), encoding="utf-8")
    return path

def jupdate(name, key, value):
    path = OUT / name
    d = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}
    d[key] = value
    path.write_text(json.dumps(d, indent=1), encoding="utf-8")

STATUS = []
STATUS_NOTE = {}   # health notes written by a stage (informational; the gates run locally)
def run_stage(name, fn):
    """Every stage: try/except, outputs written as soon as it has them, one status row appended."""
    t0 = time.time(); ok, err = True, None
    try:
        fn()
    except (Exception, SystemExit):
        ok, err = False, traceback.format_exc()
        print(f"[{name}] FAILED\n{err}")
    STATUS.append({"stage": name, "ok": ok, "error": err, "seconds": round(time.time() - t0, 1)})
    if name in STATUS_NOTE:
        STATUS[-1]["note"] = STATUS_NOTE[name]
    jdump(STATUS, "stage-status.json")
    print(f"[{name}] ok={ok} {STATUS[-1]['seconds']}s")
    gc.collect()

def find_one(pattern, what):
    hits = sorted(glob.glob(pattern, recursive=True))
    if not hits:
        raise SystemExit(f"input not found: {what} (glob {pattern}). Add it under Add Input.")
    if len(hits) > 1:
        print(f"WARNING: {len(hits)} matches for {what}; using {hits[0]}: {hits}")
    return Path(hits[0])

def sha256_file(path, chunk=1 << 24):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while blk := f.read(chunk):
            h.update(blk)
    return h.hexdigest()

def dir_bytes(path):
    return sum(p.stat().st_size for p in Path(path).rglob("*") if p.is_file())

# ---------- env-facts.json ----------
lscpu = subprocess.run(["bash", "-c", "lscpu"], capture_output=True, text=True).stdout
meminfo = Path("/proc/meminfo").read_text().splitlines()[0]
PIN_CPUS = sorted(os.sched_getaffinity(0))[:2]
jdump({
    "smoke": SMOKE,
    "lscpu_model_name": next((l.split(":", 1)[1].strip() for l in lscpu.splitlines() if l.startswith("Model name")), None),
    "lscpu": lscpu, "cpu_count": os.cpu_count(), "affinity_at_start": sorted(os.sched_getaffinity(0)),
    "pinned_cpus_for_latency": PIN_CPUS, "mem_total": meminfo,
    "thread_siblings": {c: Path(f"/sys/devices/system/cpu/cpu{c}/topology/thread_siblings_list").read_text().strip()
                        for c in sorted(os.sched_getaffinity(0)) if Path(f"/sys/devices/system/cpu/cpu{c}/topology/thread_siblings_list").exists()},
    "python": sys.version, "platform": platform.platform(),
    "torch": torch.__version__, "transformers": transformers.__version__, "onnx": onnx.__version__,
    "onnxruntime": onnxruntime.__version__, "peft": peft.__version__, "tokenizers": tokenizers.__version__,
    "numpy": np.__version__,
    "optimum": "not used: the LoRA graph is custom (two readout logits only), so the export is torch.onnx.export",
}, "env-facts.json")

# ---------- inputs, located by glob, no slug hardcoded ----------
TEST = json.loads(find_one("/kaggle/input/**/phase3-inputs-test.json", "inputs dataset (test)").read_text(encoding="utf-8"))
TV = json.loads(find_one("/kaggle/input/**/phase3-inputs-train-val.json", "inputs dataset (train-val)").read_text(encoding="utf-8"))
assert TEST["frozen_labels_sha256"] == TV["frozen_labels_sha256"]
test_rows = sorted(TEST["pairs"], key=lambda r: r["pair_id"])
tv_rows = sorted(TV["pairs"], key=lambda r: r["pair_id"])
assert len(test_rows) == 287 and len(tv_rows) == 672
assert not any(k in r for r in test_rows for k in ("label", "tier", "split")), "TEST leak!"
# TRAIN_VAL labels are used ONLY to pick which pair_ids are the 133 scored validation pairs.
val_rows = [r for r in tv_rows if r["train_or_val"] == "val" and r["label"] in ("M", "N")]
assert len(val_rows) == 133, len(val_rows)

def head(rows):
    return rows[:N_SET] if N_SET else rows

CE_SETS = {"test": head(test_rows), "trainval": head(tv_rows)}
LLM_SETS = {"val": head(val_rows), "test": head(test_rows)}
print({k: len(v) for k, v in CE_SETS.items()}, {k: len(v) for k, v in LLM_SETS.items()})

CE_DIR = find_one("/kaggle/input/**/ce-ft-best/config.json", "cross-encoder weights (retrain output ce-ft-best/)").parent
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
S = {}   # cross-stage state

In [ ]:
from typing import Any
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer

# TEMPLATE-BEGIN
# Kept byte-identical in notebooks/phase3-llm-finetune.py; checked by
# tests/test_llm_prompt_notebook_parity.py, which extracts the text strictly between the
# "# TEMPLATE-BEGIN" and "# TEMPLATE-END" marker lines in both files and asserts equality.
# Nothing outside these two marker lines is compared, so comments/docstrings elsewhere in either
# file may differ freely -- only what sits between them is the contract.

# Bumped whenever the template text changes, so a stale comparison (baseline scored under one
# template, notebook trained under a byte-different one) can never happen silently. Recorded in
# DECISIONS.md ADR-0028 addendum #21 and printed by the notebook.
LLM_PROMPT_VERSION = "llm-prompt-v1"

# Deliberately silent on tiers, split membership, label counts, or how the dataset was sampled --
# none of that is information the task itself needs, and stating it would let the model condition
# on sampling design rather than on the two listings' actual content. States the same operational
# question and rule ladder docs/learned/phase3-annotation-conventions.md defines for the human
# annotator (rules 1-6), so the model is asked the same question the labels themselves answer --
# not a looser or stricter one.
_INSTRUCTION = (
    "You are comparing two competitor-shop listings for pet food, treats or litter products.\n"
    "Decide whether Listing A and Listing B are the SAME purchasable unit: a price-comparison "
    "engine comparing their prices would be comparing the same physical product.\n"
    "\n"
    "Each of these, on its own, makes them NOT the same unit: a different net weight or volume; "
    "a different pack count (a stated count of 1 and no count stated are the SAME); a different "
    "bonus weight; a different species (dog vs cat); a different life-stage, sterilised, light, "
    "sensitive, indoor, dental, urinary or joint-care formula; a different food form (dry vs "
    "wet/tin/pouch -- wet, tin and pouch among themselves are NOT a difference); a different "
    "breed size (e.g. Mini vs Maxi); a different flavour (the same flavour word in another "
    "language, e.g. Salmon/Somon, is NOT a difference).\n"
    "\n"
    "None of these, on their own, make them different: generic descriptive words (grain-free, "
    "premium, natural, holistic); an animal-weight dosage band (e.g. 12-25 kg) stated on only "
    "one side; a mismatched brand name alone when everything else about the listing matches.\n"
    "\n"
    "Answer with exactly one word: Yes or No."
)


def build_llm_prompt(text_a: str, text_b: str) -> str:
    """Build the completion prompt; see module docstring for the design rationale."""
    return f"{_INSTRUCTION}\n\nListing A: {text_a}\nListing B: {text_b}\nAnswer:"


# TEMPLATE-END

# Readout token ids, resolved EXACTLY as notebooks/phase3-llm-finetune.py resolves them.
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def _single_token_id(word: str) -> int:
    ids = tokenizer.encode(word, add_special_tokens=False)
    if len(ids) != 1:
        raise SystemExit(
            f"REFUSING TO RUN: {word!r} encodes to {len(ids)} token(s) ({ids}), not exactly "
            "one, for this tokenizer -- the two-token readout requires both ' Yes' and ' No' to "
            "be single tokens. Falling back silently would score gibberish; fix the readout "
            "design (a different pair of words, or a different model) instead of proceeding."
        )
    return ids[0]


YES_TOKEN_ID = _single_token_id(" Yes")
NO_TOKEN_ID = _single_token_id(" No")

def llm_rows(rows):
    return [{"pair_id": r["pair_id"], "prompt": build_llm_prompt(r["text_a"], r["text_b"])} for r in rows]

def ce_rows(rows):
    return [{"pair_id": r["pair_id"], "text_a": r["text_a"], "text_b": r["text_b"]} for r in rows]

In [ ]:
# The worker is a separate process: OMP_NUM_THREADS / CPU affinity must be set BEFORE onnxruntime is
# imported, and peak RSS is only meaningful per model if the process holds nothing but that model.
# It imports onnxruntime + numpy + tokenizers only (no torch): this is the serving path.
WORKER_SRC = r"""
import json, os, sys, time
job = json.load(open(sys.argv[1]))
PIN = job["pin"]
if PIN:
    for k in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
        os.environ[k] = "2"
    cpus = sorted(os.sched_getaffinity(0))[:2]
    os.sched_setaffinity(0, cpus)
import resource
import numpy as np
import onnxruntime as ort
from tokenizers import Tokenizer

so = ort.SessionOptions()
if PIN:
    so.intra_op_num_threads = 2
    so.inter_op_num_threads = 1
    so.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
sess = ort.InferenceSession(job["model"], so, providers=["CPUExecutionProvider"])
assert [i.name for i in sess.get_inputs()] == ["input_ids", "attention_mask"], sess.get_inputs()
tok = Tokenizer.from_file(job["tokenizer"])
tok.no_padding()
KIND, PAD, MAXLEN = job["kind"], job["pad_id"], 256
if KIND == "ce":
    tok.enable_truncation(max_length=MAXLEN)

def tokenize(row):
    if KIND == "ce":
        return tok.encode(row["text_a"], row["text_b"]).ids
    ids = tok.encode(row["prompt"], add_special_tokens=False).ids
    assert len(ids) <= 1024, "prompt over the training MAX_LENGTH: refusing to truncate"
    return ids

def batch_of(id_lists):
    n = max(len(x) for x in id_lists)
    ids = np.full((len(id_lists), n), PAD, dtype=np.int64)
    mask = np.zeros((len(id_lists), n), dtype=np.int64)
    for i, x in enumerate(id_lists):
        ids[i, : len(x)] = x
        mask[i, : len(x)] = 1
    return {"input_ids": ids, "attention_mask": mask}

def readout(out):
    out = out.astype(np.float64)
    if KIND == "ce":
        return (1.0 / (1.0 + np.exp(-out[:, 0]))).tolist()
    return (1.0 / (1.0 + np.exp(out[:, 1] - out[:, 0]))).tolist()   # softmax over (Yes, No), P(Yes)

def score_one(row):
    return readout(sess.run(None, batch_of([tokenize(row)]))[0])[0]

result = {}
if job.get("score"):
    for name, spec in job["score"].items():
        rows = json.load(open(spec["pairs"]))
        scores = {r["pair_id"]: score_one(r) for r in rows}
        json.dump(scores, open(spec["out"], "w"), indent=1)
        result[name] = len(scores)

def pct(a):
    a = np.asarray(a, dtype=np.float64)
    return {"p50": float(np.percentile(a, 50)), "p95": float(np.percentile(a, 95)),
            "p99": float(np.percentile(a, 99)), "mean": float(a.mean()), "n": int(a.size)}

if job.get("bench"):
    b = job["bench"]
    rows = json.load(open(b["pairs"]))
    for r in rows[: b["n_warmup"]]:
        score_one(r)
    e2e, run_only, lens = [], [], []
    for r in rows[: b["n_lat"]]:
        t0 = time.perf_counter_ns()
        ids = tokenize(r)
        feed = batch_of([ids])
        t1 = time.perf_counter_ns()
        out = sess.run(None, feed)[0]
        t2 = time.perf_counter_ns()
        readout(out)
        t3 = time.perf_counter_ns()
        e2e.append((t3 - t0) / 1e6)
        run_only.append((t2 - t1) / 1e6)
        lens.append(len(ids))
    lat = {
        "model": job["name"], "variant": job["variant"], "pinned": PIN,
        "threads": 2 if PIN else None, "cpus": sorted(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else None,
        "ort_version": ort.__version__, "n_warmup": b["n_warmup"],
        "summary_e2e_ms": pct(e2e), "summary_run_only_ms": pct(run_only),
        "token_length": {"min": int(min(lens)), "p50": float(np.percentile(lens, 50)),
                         "p95": float(np.percentile(lens, 95)), "max": int(max(lens))},
        "samples_e2e_ms": e2e, "samples_run_only_ms": run_only, "token_lengths": lens,
    }
    rss_after_latency = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024.0
    # throughput: batches of 16, tokenize + pad + run + readout, wall clock
    bs, batch_secs = b["batch"], []
    for i in range(b["n_thr_batches"]):
        chunk = rows[i * bs : (i + 1) * bs]
        t0 = time.perf_counter()
        feed = batch_of([tokenize(r) for r in chunk])
        readout(sess.run(None, feed)[0])
        batch_secs.append(time.perf_counter() - t0)
    n_pairs = b["n_thr_batches"] * bs
    thr = {"pairs": n_pairs, "batch": bs, "batch_seconds": batch_secs,
           "pairs_per_s": n_pairs / sum(batch_secs)}
    lat["throughput"] = thr
    lat["peak_rss_mb_after_batch1_latency"] = rss_after_latency   # batch-1 serving
    lat["peak_rss_mb"] = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024.0   # after the batch-16 pass
    json.dump(lat, open(b["out"], "w"), indent=1)
    result["bench"] = {"p50_e2e_ms": lat["summary_e2e_ms"]["p50"], "pairs_per_s": thr["pairs_per_s"],
                       "peak_rss_mb": lat["peak_rss_mb"]}
print(json.dumps(result))
"""
(TMP / "worker.py").write_text(WORKER_SRC, encoding="utf-8")

def run_worker(tag, **job):
    path = TMP / f"job-{tag}.json"
    path.write_text(json.dumps(job), encoding="utf-8")
    r = subprocess.run([sys.executable, str(TMP / "worker.py"), str(path)], capture_output=True, text=True)
    print(f"[worker {tag}] rc={r.returncode} {r.stdout.strip()[-300:]}")
    if r.returncode != 0:
        raise RuntimeError(f"worker {tag} failed:\n{r.stderr[-2000:]}")

def write_rows(kind, name, rows):
    path = TMP / f"pairs-{kind}-{name}.json"
    path.write_text(json.dumps(ce_rows(rows) if kind == "ce" else llm_rows(rows)), encoding="utf-8")
    return str(path)

def score_spec(kind, sets, tag):
    """One scoring spec per set. Files: preds-<kind>-<tag>-<set>.json"""
    return {name: {"pairs": write_rows(kind, name, rows), "out": str(OUT / f"preds-{kind}-{tag}-{name}.json")}
            for name, rows in sets.items()}

def bench_spec(kind, name, variant):
    need = max(N_LAT, N_WARMUP, N_THR_BATCHES * THR_BATCH)   # test_rows is already in pair_id order
    return {"pairs": write_rows(kind, "bench", test_rows[:need]),
            "n_warmup": N_WARMUP, "n_lat": N_LAT, "n_thr_batches": N_THR_BATCHES, "batch": THR_BATCH,
            "out": str(OUT / f"latency-{name}-{variant}.json")}

def quantize(src, dst, external):
    from onnxruntime.quantization import QuantType, quantize_dynamic
    Path(dst).parent.mkdir(parents=True, exist_ok=True)
    quantize_dynamic(str(src), str(dst), weight_type=QuantType.QInt8, use_external_data_format=external)

def export_onnx(wrapper, example, path, dynamic_axes):
    """Legacy TorchScript exporter first (robust for BERT/Qwen); the dynamo exporter as fallback."""
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    kw = dict(input_names=["input_ids", "attention_mask"], output_names=["out"], dynamic_axes=dynamic_axes, opset_version=17)
    wrapper.eval()
    with torch.no_grad():
        try:
            torch.onnx.export(wrapper, example, str(path), dynamo=False, **kw)
        except Exception as e:
            print("legacy exporter failed, trying dynamo:", repr(e)[:300])
            torch.onnx.export(wrapper, example, str(path), dynamo=True, **kw)

In [ ]:
def ce_reference():
    """PyTorch fp32 CPU reference, batch 1, same tokenizer/transformers path as training."""
    facts_path = find_one("/kaggle/input/**/ce-run-facts.json", "ce-run-facts.json (retrain output)")
    weights = CE_DIR / "model.safetensors"
    w_sha = sha256_file(weights)
    expected = json.loads(facts_path.read_text())["weights_sha256"]
    jupdate("model-facts.json", "ce_weights", {"sha256": w_sha, "expected_sha256": expected, "match": w_sha == expected})
    if w_sha != expected:
        raise SystemExit("cross-encoder weights do not match ce-run-facts.json")
    tok = AutoTokenizer.from_pretrained(CE_DIR)
    model = AutoModelForSequenceClassification.from_pretrained(CE_DIR).eval()
    S["ce_pad_id"] = tok.pad_token_id
    jupdate("model-facts.json", "ce_param_count", sum(p.numel() for p in model.parameters()))
    for name, rows in CE_SETS.items():
        out = {}
        for r in rows:
            enc = tok(r["text_a"], r["text_b"], truncation=True, max_length=256, return_tensors="pt")
            with torch.no_grad():
                out[r["pair_id"]] = float(torch.sigmoid(model(**enc).logits[:, 0].float())[0])
        jdump(out, f"preds-ce-ptfp32-{name}.json")

def ce_export():
    tok = AutoTokenizer.from_pretrained(CE_DIR)
    model = AutoModelForSequenceClassification.from_pretrained(CE_DIR, attn_implementation="eager").eval()
    class CEWrap(torch.nn.Module):
        def __init__(self, m):
            super().__init__(); self.m = m
        def forward(self, input_ids, attention_mask):
            return self.m(input_ids=input_ids, attention_mask=attention_mask).logits
    rows = CE_SETS["test"][:2]
    enc = tok([r["text_a"] for r in rows], [r["text_b"] for r in rows], truncation=True, max_length=256,
              padding=True, return_tensors="pt")
    assert "token_type_ids" not in enc, "tokenizer emits token_type_ids: the exported graph would ignore them"
    S["ce_pad_id"] = tok.pad_token_id
    path = TMP / "ce-fp32" / "model.onnx"
    export_onnx(CEWrap(model), (enc["input_ids"], enc["attention_mask"]), path,
                {"input_ids": {0: "b", 1: "s"}, "attention_mask": {0: "b", 1: "s"}, "out": {0: "b"}})
    S["ce_fp32"] = path
    jupdate("model-facts.json", "ce_onnx_fp32", {"bytes": dir_bytes(path.parent), "sha256": sha256_file(path)})
    tok.save_pretrained(TMP / "ce-tok")

def ce_int8():
    dst = TMP / "ce-int8" / "model.onnx"
    quantize(S["ce_fp32"], dst, external=False)
    S["ce_int8"] = dst
    jupdate("model-facts.json", "ce_onnx_int8", {"bytes": dir_bytes(dst.parent), "sha256": sha256_file(dst),
                                                "quantization": "onnxruntime quantize_dynamic, QInt8 weights, per-tensor"})

def ce_score():
    for variant, key in (("onnxfp32", "ce_fp32"), ("int8", "ce_int8")):
        run_worker(f"ce-score-{variant}", kind="ce", name="ce", variant=variant, model=str(S[key]),
                   tokenizer=str(TMP / "ce-tok" / "tokenizer.json"), pad_id=S["ce_pad_id"], pin=False,
                   score=score_spec("ce", CE_SETS, variant))
    # health note only (not a gate; the gates run locally in session 2)
    a = json.loads((OUT / "preds-ce-ptfp32-test.json").read_text())
    b = json.loads((OUT / "preds-ce-onnxfp32-test.json").read_text())
    STATUS_NOTE["CE-score"] = f"max|ptfp32-onnxfp32| on test = {max(abs(a[k] - b[k]) for k in a):.2e}"

def ce_latency():
    for variant, key in (("onnxfp32", "ce_fp32"), ("int8", "ce_int8")):
        run_worker(f"ce-bench-{variant}", kind="ce", name="ce", variant=variant, model=str(S[key]),
                   tokenizer=str(TMP / "ce-tok" / "tokenizer.json"), pad_id=S["ce_pad_id"], pin=True,
                   bench=bench_spec("ce", "ce", variant))
        lat = json.loads((OUT / f"latency-ce-{variant}.json").read_text())
        jupdate("throughput.json", f"ce-{variant}", lat["throughput"])
        jupdate("model-facts.json", f"ce_{variant}_peak_rss_mb", lat["peak_rss_mb"])

run_stage("CE-reference", ce_reference)
run_stage("CE-export", ce_export)
run_stage("CE-int8", ce_int8)
run_stage("CE-score", ce_score)
run_stage("CE-latency", ce_latency)

In [ ]:
def llm_merge():
    """Find the adapter, prove which folder is which, merge into the fp32 base."""
    cfgs = sorted(glob.glob("/kaggle/input/**/adapter_config.json", recursive=True))
    dirs = {Path(c).parent.name: Path(c).parent for c in cfgs}
    print("adapter folders:", {k: str(v) for k, v in dirs.items()})
    hashes = {k: sha256_file(v / "adapter_model.safetensors") for k, v in dirs.items()}
    jupdate("model-facts.json", "lora_adapter_dirs", {k: {"path": str(v), "sha256": hashes[k]} for k, v in dirs.items()})
    if not dirs:
        raise SystemExit("no adapter_config.json under /kaggle/input")
    if len(set(hashes.values())) > 1:
        raise SystemExit(f"adapter folders differ, identity unresolved: {hashes}")
    adapter_dir = dirs.get("adapter") or dirs.get("adapter-epoch-8") or next(iter(dirs.values()))
    try:
        base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.float32)
    except TypeError:
        base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float32)
    from peft import PeftModel
    merged = PeftModel.from_pretrained(base, str(adapter_dir)).merge_and_unload().eval()
    assert next(merged.parameters()).dtype == torch.float32
    dst = TMP / "llm-merged"
    merged.save_pretrained(dst)
    tokenizer.save_pretrained(dst)
    jupdate("model-facts.json", "llm_param_count_merged", sum(p.numel() for p in merged.parameters()))
    jupdate("model-facts.json", "llm_adapter_used", str(adapter_dir))
    jupdate("model-facts.json", "llm_yes_no_token_ids", [YES_TOKEN_ID, NO_TOKEN_ID])
    S["llm_merged"] = dst

def llm_reference():
    """PyTorch fp32 CPU, full-vocab forward, the training notebook's score_pair readout."""
    model = AutoModelForCausalLM.from_pretrained(S["llm_merged"], dtype=torch.float32).eval()
    for name, rows in LLM_SETS.items():
        out = {}
        for r in rows:
            ids = tokenizer(build_llm_prompt(r["text_a"], r["text_b"]), return_tensors="pt", add_special_tokens=False).input_ids
            with torch.no_grad():
                logits = model(input_ids=ids).logits[0, -1, :]
            two = torch.stack([logits[YES_TOKEN_ID], logits[NO_TOKEN_ID]])
            out[r["pair_id"]] = float(F.softmax(two.float(), dim=0)[0].item())
        jdump(out, f"preds-llm-ptfp32-{name}.json")

def llm_export():
    model = AutoModelForCausalLM.from_pretrained(S["llm_merged"], dtype=torch.float32, attn_implementation="eager").eval()
    class LLMWrap(torch.nn.Module):
        """Transformer body + ONLY the two readout logits at the last real prompt token.
        No KV-cache, one forward pass, no full-vocab logits (protocol 5.2)."""
        def __init__(self, lm, yes_id, no_id):
            super().__init__()
            self.body = lm.model
            self.register_buffer("head", lm.lm_head.weight[[yes_id, no_id]].detach().clone())
        def forward(self, input_ids, attention_mask):
            h = self.body(input_ids=input_ids, attention_mask=attention_mask, use_cache=False).last_hidden_state
            last = (attention_mask.sum(dim=1) - 1).view(-1, 1, 1).expand(-1, 1, h.size(-1))
            return h.gather(1, last).squeeze(1) @ self.head.T
    rows = LLM_SETS["test"][:2]
    enc = [tokenizer(build_llm_prompt(r["text_a"], r["text_b"]), add_special_tokens=False)["input_ids"] for r in rows]
    n = max(len(x) for x in enc)
    ids = torch.tensor([x + [tokenizer.pad_token_id] * (n - len(x)) for x in enc])
    mask = torch.tensor([[1] * len(x) + [0] * (n - len(x)) for x in enc])
    tmp_path = TMP / "llm-fp32-raw" / "model.onnx"
    export_onnx(LLMWrap(model, YES_TOKEN_ID, NO_TOKEN_ID), (ids, mask), tmp_path,
                {"input_ids": {0: "b", 1: "s"}, "attention_mask": {0: "b", 1: "s"}, "out": {0: "b"}})
    # consolidate into ONE external-data file so the graph is a single small .onnx + one .data
    path = TMP / "llm-fp32" / "model.onnx"
    path.parent.mkdir(parents=True, exist_ok=True)
    m = onnx.load(str(tmp_path))
    onnx.save_model(m, str(path), save_as_external_data=True, all_tensors_to_one_file=True,
                    location="model.onnx.data", size_threshold=1024)
    del m
    shutil.rmtree(tmp_path.parent, ignore_errors=True)
    S["llm_fp32"] = path
    jupdate("model-facts.json", "llm_onnx_fp32", {"bytes": dir_bytes(path.parent), "sha256_graph": sha256_file(path),
                                                 "sha256_data": sha256_file(path.parent / "model.onnx.data")})

def llm_int8():
    dst = TMP / "llm-int8" / "model.onnx"
    quantize(S["llm_fp32"], dst, external=True)
    S["llm_int8"] = dst
    jupdate("model-facts.json", "llm_onnx_int8", {"bytes": dir_bytes(dst.parent), "sha256_graph": sha256_file(dst),
                                                 "quantization": "onnxruntime quantize_dynamic, QInt8 weights, per-tensor; embedding Gather stays fp32"})

def llm_score():
    for variant, key in (("onnxfp32", "llm_fp32"), ("int8", "llm_int8")):
        run_worker(f"llm-score-{variant}", kind="llm", name="llm", variant=variant, model=str(S[key]),
                   tokenizer=str(S["llm_merged"] / "tokenizer.json"), pad_id=tokenizer.pad_token_id, pin=False,
                   score=score_spec("llm", LLM_SETS, variant))
    a = json.loads((OUT / "preds-llm-ptfp32-test.json").read_text())
    b = json.loads((OUT / "preds-llm-onnxfp32-test.json").read_text())
    STATUS_NOTE["LLM-score"] = f"max|ptfp32-onnxfp32| on test = {max(abs(a[k] - b[k]) for k in a):.2e}"

def llm_latency():
    for variant, key in (("onnxfp32", "llm_fp32"), ("int8", "llm_int8")):
        run_worker(f"llm-bench-{variant}", kind="llm", name="llm", variant=variant, model=str(S[key]),
                   tokenizer=str(S["llm_merged"] / "tokenizer.json"), pad_id=tokenizer.pad_token_id, pin=True,
                   bench=bench_spec("llm", "llm", variant))
        lat = json.loads((OUT / f"latency-llm-{variant}.json").read_text())
        jupdate("throughput.json", f"llm-{variant}", lat["throughput"])
        jupdate("model-facts.json", f"llm_{variant}_peak_rss_mb", lat["peak_rss_mb"])

run_stage("LLM-merge", llm_merge)
run_stage("LLM-reference", llm_reference)
run_stage("LLM-export", llm_export)
run_stage("LLM-int8", llm_int8)
run_stage("LLM-score", llm_score)
run_stage("LLM-latency", llm_latency)

In [ ]:
jdump(STATUS, "stage-status.json")
print(json.dumps(STATUS, indent=1))
print("SMOKE" if SMOKE else "FULL", "run finished. Outputs under", OUT)
from IPython.display import FileLink, display
for p in sorted(OUT.rglob("*.json")):
    display(FileLink(str(p.relative_to(WORK))))